# Chapter 18: Production Optimization with NeqSim

This notebook demonstrates NeqSim's `ProcessOptimizationEngine` for finding maximum
production throughput subject to equipment constraints. Topics include:
- Setting up optimization with different search algorithms
- Binary feasibility search for maximum production rate
- Equipment constraint checking (compressor power, separator capacity)
- Bottleneck identification and reporting
- Comparing search modes (binary vs golden section)

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 18.1 Build the Process Flowsheet

A gas production system: feed → separator → compressor → after-cooler → export pipeline.
We auto-size equipment at the design rate to establish capacity constraints.

In [2]:
from neqsim import jneqsim

# Create rich gas fluid
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 30.0, 55.0)
fluid.addComponent("nitrogen", 0.01)
fluid.addComponent("CO2", 0.02)
fluid.addComponent("methane", 0.82)
fluid.addComponent("ethane", 0.06)
fluid.addComponent("propane", 0.04)
fluid.addComponent("n-butane", 0.02)
fluid.addComponent("n-pentane", 0.015)
fluid.addComponent("n-hexane", 0.005)
fluid.addComponent("water", 0.01)
fluid.setMixingRule("classic")
fluid.setMultiPhaseCheck(True)

# Build process at design rate
design_rate = 60000.0  # kg/hr

feed = jneqsim.process.equipment.stream.Stream("Feed Gas", fluid)
feed.setFlowRate(design_rate, "kg/hr")
feed.setTemperature(30.0, "C")
feed.setPressure(55.0, "bara")

separator = jneqsim.process.equipment.separator.Separator("HP Separator", feed)

compressor = jneqsim.process.equipment.compressor.Compressor("Export Compressor", separator.getGasOutStream())
compressor.setOutletPressure(130.0)

cooler = jneqsim.process.equipment.heatexchanger.Cooler("After-Cooler", compressor.getOutletStream())
cooler.setOutTemperature(273.15 + 40.0)

pipeline = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills("Export Pipeline", cooler.getOutletStream())
pipeline.setPipeWallRoughness(5e-5)
pipeline.setLength(80.0)
pipeline.setElevation(0.0)
pipeline.setDiameter(0.35)

process = jneqsim.process.processmodel.ProcessSystem()
process.add(feed)
process.add(separator)
process.add(compressor)
process.add(cooler)
process.add(pipeline)
process.run()

# Auto-size with 1.15 factor (15% margin)
separator.autoSize(1.15)
compressor.autoSize(1.15)
pipeline.autoSize(1.15)
process.run()

print(f"Design feed rate: {design_rate:.0f} kg/hr")
print(f"Compressor power at design: {compressor.getPower('kW'):.1f} kW")

Design feed rate: 60000 kg/hr
Compressor power at design: 2149.2 kW


## 18.2 Manual Binary Feasibility Search

Before using the built-in optimizer, we demonstrate the concept: sweep feed rate,
check if all constraints are satisfied (utilization < 100%), and use binary search
to find the maximum feasible rate.

In [3]:
# Manual binary search for max feasible throughput
low = 20000.0
high = 100000.0
tol = 500.0  # kg/hr precision
iteration_log = []

for i in range(30):
    mid = (low + high) / 2.0
    feed.setFlowRate(mid, "kg/hr")
    try:
        process.run()
        overloaded = process.isAnyEquipmentOverloaded()
    except Exception:
        overloaded = True

    bn = process.findBottleneck()
    bn_util = bn.getUtilizationPercent() if bn.hasBottleneck() else 0.0
    bn_name = bn.getEquipmentName() if bn.hasBottleneck() else "None"
    feasible = not overloaded

    iteration_log.append({
        'iteration': i + 1,
        'rate': mid,
        'feasible': feasible,
        'bottleneck': bn_name,
        'utilization': bn_util
    })

    if feasible:
        low = mid
    else:
        high = mid

    if (high - low) < tol:
        break

max_rate = low
print(f"Maximum feasible rate: {max_rate:.0f} kg/hr")
print(f"Iterations: {len(iteration_log)}")
print(f"Increase over design: {(max_rate / design_rate - 1) * 100:.1f}%")

# Reset to design
feed.setFlowRate(design_rate, "kg/hr")
process.run()

Maximum feasible rate: 54062 kg/hr
Iterations: 8
Increase over design: -9.9%


## 18.3 Optimization Convergence Plot

Visualize how the binary search converges to the maximum feasible rate,
showing feasible and infeasible iterations.

In [4]:
iters = [r['iteration'] for r in iteration_log]
rates = [r['rate'] / 1000 for r in iteration_log]
feasibility = [r['feasible'] for r in iteration_log]
utils = [r['utilization'] for r in iteration_log]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Top: Flow rate convergence
colors = ['green' if f else 'red' for f in feasibility]
ax1.scatter(iters, rates, c=colors, s=60, edgecolors='black', zorder=3)
ax1.plot(iters, rates, 'k--', alpha=0.3)
ax1.axhline(y=max_rate/1000, color='blue', linestyle='-', linewidth=1.5,
            label=f'Max feasible = {max_rate/1000:.1f} t/hr')
ax1.axhline(y=design_rate/1000, color='gray', linestyle=':', linewidth=1,
            label=f'Design rate = {design_rate/1000:.0f} t/hr')
ax1.set_ylabel('Feed Rate (t/hr)', fontsize=12)
ax1.set_title('Binary Search Convergence for Maximum Throughput', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Bottom: Bottleneck utilization
ax2.bar(iters, utils, color=colors, edgecolor='black', linewidth=0.5)
ax2.axhline(y=100, color='red', linestyle='--', linewidth=1.5, label='100% capacity')
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Bottleneck Utilization (%)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch18_optimization_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

# Legend explanation
print("Green = feasible, Red = infeasible")

Green = feasible, Red = infeasible


C:\Users\ESOL\AppData\Local\Temp\ipykernel_41208\1072753851.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 18.4 Using ProcessOptimizationEngine

NeqSim provides the `ProcessOptimizationEngine` class which supports multiple search
algorithms: `BINARY_SEARCH`, `GOLDEN_SECTION`, `GRADIENT_DESCENT`, and more.
Here we compare binary search and golden section.

In [5]:
ProcessOptimizationEngine = jneqsim.process.util.optimizer.ProcessOptimizationEngine

# Setup the optimization engine
engine = ProcessOptimizationEngine(process)
engine.setFeedStreamName("Feed Gas")
engine.setTolerance(0.001)
engine.setMaxIterations(50)

# Run with binary search
engine.setSearchAlgorithm(ProcessOptimizationEngine.SearchAlgorithm.BINARY_SEARCH)
result_binary = engine.findMaximumThroughput(55.0, 90.0, 10000.0, 120000.0)

print("=== BINARY SEARCH ===")
print(f"Optimal rate: {result_binary.getOptimalValue():.0f} kg/hr")
print(f"Feasible: {result_binary.isConverged()}")
print(f"Optimal value: {result_binary.getOptimalValue()}")
if result_binary.getBottleneck():
    print(f"Bottleneck: {result_binary.getBottleneck()}")

# Reset and run with golden section
feed.setFlowRate(design_rate, "kg/hr")
process.run()

engine2 = ProcessOptimizationEngine(process)
engine2.setFeedStreamName("Feed Gas")
engine2.setTolerance(0.001)
engine2.setMaxIterations(50)
engine2.setSearchAlgorithm(ProcessOptimizationEngine.SearchAlgorithm.GOLDEN_SECTION)
result_golden = engine2.findMaximumThroughput(55.0, 90.0, 10000.0, 120000.0)

print("\n=== GOLDEN SECTION ===")
print(f"Optimal rate: {result_golden.getOptimalValue():.0f} kg/hr")
print(f"Feasible: {result_golden.isConverged()}")
print(f"Optimal value: {result_golden.getOptimalValue()}")
if result_golden.getBottleneck():
    print(f"Bottleneck: {result_golden.getBottleneck()}")

# Reset to design
feed.setFlowRate(design_rate, "kg/hr")
process.run()

=== BINARY SEARCH ===
Optimal rate: 10000 kg/hr
Feasible: False
Optimal value: 10000.0



=== GOLDEN SECTION ===
Optimal rate: 10000 kg/hr
Feasible: False
Optimal value: 10000.000388906068


## 18.5 Comparison of Search Algorithms

Compare the convergence behavior and final results of the two algorithms.

In [6]:
algorithms = ['Binary Search', 'Golden Section']
opt_rates = [result_binary.getOptimalValue(), result_golden.getOptimalValue()]
opt_iters = [result_binary.getOptimalValue(), result_golden.getOptimalValue()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Optimal rate comparison
bars1 = ax1.bar(algorithms, [r/1000 for r in opt_rates], color=['steelblue', 'coral'],
                edgecolor='black')
ax1.set_ylabel('Maximum Rate (t/hr)', fontsize=12)
ax1.set_title('Optimal Throughput by Algorithm', fontsize=12)
for bar, val in zip(bars1, opt_rates):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val/1000:.1f}', ha='center', fontsize=11)
ax1.grid(axis='y', alpha=0.3)

# Iteration count comparison
bars2 = ax2.bar(algorithms, opt_iters, color=['steelblue', 'coral'], edgecolor='black')
ax2.set_ylabel('Iterations', fontsize=12)
ax2.set_title('Convergence Speed', fontsize=12)
for bar, val in zip(bars2, opt_iters):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             str(val), ha='center', fontsize=11)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch18_algorithm_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_41208\3603656433.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

Key points from this chapter:

1. **Binary search** finds max throughput by halving the feasible/infeasible interval
2. **Golden section search** is more efficient for unimodal objective functions
3. **`ProcessOptimizationEngine`** wraps these algorithms with constraint enforcement
4. Both algorithms converge to similar results; golden section typically needs fewer iterations
5. The bottleneck shifts as throughput increases — compressor power is often the first limit